# Feature Creator — All Cities (LaDe Dataset)

**Cities:** Shanghai · Hangzhou · Chongqing  
**Target:** `eta_mins = (sign_time − receipt_time) / 60`  
**Author:** Soumya · Thesis: Causal-informed RL for ETA Prediction

---

## Pipeline Overview

```
load_delivery()
    └─ load_gps_window()
        └─ courier_snapshot()  →  filter_stale_gps()
            └─ compute_speed_percentile()
                └─ compute_trajectory_features()
add_euclidean_distance()
add_batch_features()
add_workload()
add_temporal_features()
add_workload_nonlinear()
add_gps_missingness_flag()
add_typecode_encoding()
add_spatial_congestion()
add_weather()
final_sanity_check()
```

Each section below is a reusable function. The **Run All Cities** section at the
bottom loops over all three cities and saves per-city CSVs plus an optional
combined parquet.


## 0. Install & Imports

In [ ]:
!pip install duckdb polars pyarrow --quiet

In [ ]:
import polars as pl
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import date

pl.Config.set_tbl_rows(10)
print("Polars:", pl.__version__, " | DuckDB:", duckdb.__version__)


## 1. Configuration

Edit the paths and parameters below before running.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE       = "/content/drive/MyDrive/ml/PROCESSED/matched/city_divided/"
WEATHER_BASE = "/content/drive/MyDrive/ml/weather-outputs/"
OUTPUT_DIR = "/content/drive/MyDrive/ml/PROCESSED/final/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Pipeline hyper-parameters ─────────────────────────────────────────────────
PRE_MIN    = 15     # minutes of GPS history to look back before receipt_time
MAX_GAP    = 30     # GPS matches older than this (minutes) are invalidated
GRID_SIZE  = 500    # spatial grid cell size (affine coordinate units ≈ 500 m)

# ── City registry ─────────────────────────────────────────────────────────────
# Add / remove cities here. Each entry:
#   city_en       : human-readable name (matches CITY_IN_QUES in weather CSVs)
#   delivery_file : CSV filename under BASE
#   gps_file      : parquet filename under BASE
#   weather_file  : CSV filename under WEATHER_BASE
#   holidays      : list of 'YYYY-MM-DD' strings (Qingming 2021)
#   holiday_eve   : day before first holiday
CITY_CONFIGS = [
    {
        "city_en"      : "Shanghai",
        "delivery_file": "shanghai_data.csv",
        "gps_file"     : "shanghai_delivery_data.parquet",
        "weather_file" : "shanghai_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
    {
        "city_en"      : "Hangzhou",
        "delivery_file": "hangzhou_data.csv",
        "gps_file"     : "hangzhou_delivery_data.parquet",
        "weather_file" : "hangzhou_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
    {
        "city_en"      : "Chongqing",
        "delivery_file": "chongqing_data.csv",
        "gps_file"     : "chongqing_delivery_data.parquet",
        "weather_file" : "chongqing_wsi_mar17_apr20_2021.csv",
        "holidays"     : ["2021-04-03", "2021-04-04", "2021-04-05"],
        "holiday_eve"  : "2021-04-02",
    },
]

SELECTED_FEATURES = [
    "order_id", "city",
    "batch_size", "batch_rank_capped", "workload_capped", "delivery_sequence",
    "speed_mean", "distance_travelled", "gps_points", "gps_gap_min",
    "is_trajectory_available",
    "pickup_destination_distance",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "is_holiday", "is_holiday_eve", "is_weekend",
    "WSI", "precipitation", "temperature_2m", "windspeed_10m",
    "typecode_grouped_type 1", "typecode_grouped_type 2",
    "typecode_grouped_type 3", "typecode_cb",
    "spatial_congestion_norm", "courier_local_load",
    "eta_mins",
]


## 2. Pipeline Functions

All transformation logic lives here. Functions are stateless — they receive a
dataframe (and sometimes a DuckDB connection / path) and return a new dataframe.


### 2.1 `load_delivery` — Load and normalise the delivery CSV

In [ ]:
def load_delivery(csv_path: str) -> pl.DataFrame:
    """
    Load the city delivery CSV, parse datetimes, drop redundant columns,
    rename the target variable, and sort for downstream ASOF joins.

    Returns
    -------
    pl.DataFrame  sorted by (delivery_user_id, receipt_time)
    """
    df = (
        pl.read_csv(csv_path)
        .with_columns([
            pl.col("receipt_time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
            pl.col("sign_time").str.to_datetime("%Y-%m-%d %H:%M:%S"),
        ])
        .drop(["sign_lat", "sign_lng"])          # redundant destination cols
        .rename({"horizon_ETA": "eta_mins"})     # canonical target name
        .with_columns(pl.col("delivery_user_id").cast(pl.Utf8))
        .sort(["delivery_user_id", "receipt_time"])
    )
    print(f"  Loaded {df.height:,} deliveries | {df['delivery_user_id'].n_unique()} couriers")
    return df


### 2.2 `load_gps_window` — Fetch relevant GPS records via DuckDB

In [ ]:
def load_gps_window(
    con: duckdb.DuckDBPyConnection,
    gps_path: str,
    couriers: list,
    start_time,
    end_time,
    pre_min: int = PRE_MIN,
) -> pl.DataFrame:
    """
    Pull GPS rows for the relevant couriers and time range from parquet.
    Uses DuckDB for push-down filtering — avoids loading the full GPS file.

    Returns
    -------
    pl.DataFrame  columns: delivery_user_id, gps_time, lat, lng
                  sorted by (delivery_user_id, gps_time)
    """
    # DuckDB requires a tuple with ≥2 elements for the IN clause
    couriers_tuple = tuple(couriers) if len(couriers) > 1 else (couriers[0], couriers[0])

    gps_df = con.execute(f"""
        SELECT postman_id, gps_time, lat, lng
        FROM parquet_scan('{gps_path}')
        WHERE postman_id IN {couriers_tuple}
          AND gps_time BETWEEN
              TIMESTAMP '{start_time}' - INTERVAL '{pre_min} minutes'
              AND TIMESTAMP '{end_time}'
    """).pl()

    gps_df = (
        gps_df
        .rename({"postman_id": "delivery_user_id"})
        .sort(["delivery_user_id", "gps_time"])
    )
    print(f"  GPS rows loaded: {gps_df.height:,}")
    return gps_df


### 2.3 `courier_snapshot` + `filter_stale_gps` — Last known position

In [ ]:
def courier_snapshot(delivery: pl.DataFrame, gps_df: pl.DataFrame) -> pl.DataFrame:
    """
    ASOF backward join: for each receipt_time, attach the courier's most recent
    GPS point at or before that moment.

    Returns
    -------
    pl.DataFrame  delivery + last_x, last_y, last_gps_time, gps_gap_min
    """
    state = delivery.join_asof(
        gps_df,
        left_on="receipt_time",
        right_on="gps_time",
        by="delivery_user_id",
        strategy="backward",
    ).rename({"lat": "last_x", "lng": "last_y", "gps_time": "last_gps_time"})

    state = state.with_columns(
        (pl.col("receipt_time") - pl.col("last_gps_time"))
        .dt.total_minutes()
        .alias("gps_gap_min")
    )
    return state


def filter_stale_gps(state: pl.DataFrame, max_gap: int = MAX_GAP) -> pl.DataFrame:
    """
    Invalidate GPS matches older than max_gap minutes.
    Sets gps_gap_min, last_x, last_y to null for stale matches.

    Rationale: GPS older than 30 min is operationally unreliable; treating it
    as missing is safer than propagating stale location estimates.

    Returns
    -------
    pl.DataFrame  state with stale GPS columns nulled out
    """
    state = state.with_columns(
        pl.when(pl.col("gps_gap_min") <= max_gap)
          .then(pl.col("gps_gap_min"))
          .otherwise(None)
          .alias("gps_gap_min")
    )
    state = state.with_columns([
        pl.when(pl.col("gps_gap_min").is_null()).then(None).otherwise(pl.col("last_x")).alias("last_x"),
        pl.when(pl.col("gps_gap_min").is_null()).then(None).otherwise(pl.col("last_y")).alias("last_y"),
    ])

    total   = state.height
    missing = state["gps_gap_min"].null_count()
    print(f"  GPS coverage after {max_gap}-min filter: "
          f"{total - missing:,}/{total:,} ({(total-missing)/total*100:.1f}%)")
    return state


### 2.4 `compute_speed_percentile` — 99th-percentile speed cap

In [ ]:
def compute_speed_percentile(
    con: duckdb.DuckDBPyConnection,
    gps_path: str,
    percentile: float = 0.99,
) -> float:
    """
    Compute a robust speed cap from the full GPS parquet.

    Filters:
      dt > 5 s    — removes GPS duplicates (sampling ≈ 20 s)
      dt < 600 s  — removes long gaps that create phantom distance
      dist < 10000 — removes privacy-related coordinate jumps

    Returns
    -------
    float  speed threshold in affine coordinate units / second
    """
    speed_q = con.execute(f"""
        WITH motion AS (
            SELECT
                SQRT(POWER(lat - LAG(lat) OVER w, 2) +
                     POWER(lng - LAG(lng) OVER w, 2)) AS dist,
                EXTRACT(EPOCH FROM (gps_time - LAG(gps_time) OVER w)) AS dt
            FROM parquet_scan('{gps_path}')
            WINDOW w AS (PARTITION BY postman_id ORDER BY gps_time)
        ),
        clean AS (
            SELECT dist / dt AS speed
            FROM motion
            WHERE dt > 5 AND dt < 600 AND dist < 10000
        )
        SELECT quantile_cont(speed, {percentile}) FROM clean
    """).fetchone()[0]

    print(f"  Speed {percentile*100:.0f}th percentile cap: {speed_q:.4f} units/s")
    return speed_q


### 2.5 `compute_trajectory_features` — 15-min pre-delivery GPS window

In [ ]:
def compute_trajectory_features(
    con: duckdb.DuckDBPyConnection,
    gps_path: str,
    speed_cap: float,
    pre_min: int = PRE_MIN,
) -> pl.DataFrame:
    """
    For each order, aggregate GPS segments in [receipt_time - pre_min, receipt_time].

    Speed filtering (applied to each segment):
      dt ≤ 1 s     — GPS duplicate / noise
      dist > 5000  — privacy jump (affine coordinate teleportation)
      dist/dt > speed_cap — physically impossible speed (99th-pct threshold)

    Returns
    -------
    pl.DataFrame  columns: order_id, gps_points, speed_mean, distance_travelled
    """
    traj = con.execute(f"""
        WITH gps_window AS (
            SELECT
                d.order_id,
                g.gps_time, g.lat, g.lng,
                LAG(g.lat)      OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_lat,
                LAG(g.lng)      OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_lng,
                LAG(g.gps_time) OVER (PARTITION BY d.order_id ORDER BY g.gps_time) AS prev_time
            FROM delivery_tbl d
            JOIN parquet_scan('{gps_path}') g
              ON g.postman_id = d.delivery_user_id
            WHERE g.gps_time BETWEEN
                d.receipt_time - INTERVAL '{pre_min} minutes'
                AND d.receipt_time
        ),
        motion AS (
            SELECT
                order_id,
                SQRT(POWER(lat - prev_lat, 2) + POWER(lng - prev_lng, 2)) AS dist,
                EXTRACT(EPOCH FROM (gps_time - prev_time)) AS dt
            FROM gps_window
        ),
        speed_calc AS (
            SELECT
                order_id, dist, dt,
                CASE
                    WHEN dt IS NULL       THEN NULL   -- first row in window
                    WHEN dt <= 1          THEN NULL   -- GPS duplicate / noise
                    WHEN dist > 5000      THEN NULL   -- privacy coord jump
                    WHEN dist / dt > {speed_cap} THEN NULL  -- speed cap
                    ELSE dist / dt
                END AS speed
            FROM motion
        )
        SELECT
            order_id,
            COUNT(*)        AS gps_points,
            AVG(speed)      AS speed_mean,
            SUM(dist)       AS distance_travelled
        FROM speed_calc
        GROUP BY order_id
    """).pl()

    print(f"  Trajectory features computed for {traj.height:,} orders")
    return traj


### 2.6 `add_euclidean_distance` — Pickup → destination distance

In [ ]:
def add_euclidean_distance(delivery: pl.DataFrame) -> pl.DataFrame:
    """
    Euclidean distance between pickup (receipt) and destination (POI)
    in the affine coordinate space (≈ metres at ~1e6 scale).

    Adds column: pickup_destination_distance
    """
    return delivery.with_columns(
        ((pl.col("receipt_lng") - pl.col("poi_lng")).pow(2) +
         (pl.col("receipt_lat") - pl.col("poi_lat")).pow(2)).sqrt()
        .alias("pickup_destination_distance")
    )


### 2.7 `add_batch_features` — Batch size and intra-batch rank

In [ ]:
def add_batch_features(delivery: pl.DataFrame) -> pl.DataFrame:
    """
    Compute batch_size and batch_rank for simultaneous dispatcher pushes.

    Batch = set of orders sharing the same (delivery_user_id, receipt_time).
    order_id is the tiebreaker when multiple orders arrive at the exact same
    timestamp — this is critical for deterministic, reproducible ranks.

    Adds columns: batch_size, batch_rank
    """
    delivery = delivery.sort(["delivery_user_id", "receipt_time", "order_id"])

    delivery = delivery.with_columns(
        pl.len().over(["delivery_user_id", "receipt_time"]).alias("batch_size")
    )
    delivery = delivery.with_columns(
        pl.int_range(0, pl.len())
          .over(["delivery_user_id", "receipt_time"])
          .alias("batch_rank")
    )
    n_batched = delivery.filter(pl.col("batch_size") > 1).height
    print(f"  Orders in multi-order batches: {n_batched:,} "
          f"| max batch_size: {delivery['batch_size'].max()}")
    return delivery


### 2.8 `add_workload` — Concurrent active orders (event-sweep)

In [ ]:
def add_workload(delivery: pl.DataFrame) -> pl.DataFrame:
    """
    Compute active_orders_at_receipt_time via a linear-time event-sweep
    algorithm (O(n log n) sort + O(n) scan).

    Why not self-join?  A self-join is O(n²) — prohibitive at 40k+ orders.

    Algorithm:
        start_events (+1, priority=1) at receipt_time
        end_events   (-1, priority=0) at sign_time   [resolve before receipts]
        cumsum(delta) over (courier, time, priority) → active_orders
        ASOF join back to delivery on receipt_time → subtract self-count

    Adds column: active_orders_at_receipt_time
    """
    delivery = delivery.sort(["delivery_user_id", "receipt_time"])

    start_events = delivery.select([
        "delivery_user_id",
        pl.col("receipt_time").alias("time"),
        pl.lit(1).alias("delta"),
        pl.lit(1).alias("priority"),   # receipts resolve after sign events
    ])
    end_events = delivery.select([
        "delivery_user_id",
        pl.col("sign_time").alias("time"),
        pl.lit(-1).alias("delta"),
        pl.lit(0).alias("priority"),   # sign events resolve first
    ])

    events = (
        pl.concat([start_events, end_events])
        .sort(["delivery_user_id", "time", "priority"])
        .with_columns(
            pl.col("delta").cum_sum().over("delivery_user_id").alias("active_orders")
        )
    )

    # ASOF: for each receipt_time, grab the cumulative count just before/at it
    workload = delivery.join_asof(
        events.select(["delivery_user_id", "time", "active_orders"]),
        left_on="receipt_time",
        right_on="time",
        by="delivery_user_id",
        strategy="backward",
    )

    # Subtract self — the current order is included in the sweep count
    workload = workload.with_columns(
        (pl.col("active_orders") - 1).clip(lower_bound=0)
        .alias("active_orders_at_receipt_time")
    )

    print(f"  Workload | mean: {workload['active_orders_at_receipt_time'].mean():.2f} "
          f"| max: {workload['active_orders_at_receipt_time'].max()}")
    return workload


### 2.9 `add_temporal_features` — Cyclical time + calendar flags

In [ ]:
def add_temporal_features(
    features: pl.DataFrame,
    holidays: list,
    holiday_eve: str,
) -> pl.DataFrame:
    """
    Adds time-of-day and calendar context features.

    Cyclical encoding preserves the circular topology of time:
        sin/cos(2π × hour / 24) — avoids the 23→0 discontinuity.

    Adds columns:
        hour, weekday, day_of_week
        hour_sin, hour_cos, day_sin, day_cos
        is_weekend, is_holiday, is_holiday_eve
        delivery_sequence  (cumulative per courier — fatigue proxy)
    """
    features = features.with_columns([
        pl.col("receipt_time").dt.hour().alias("hour"),
        pl.col("receipt_time").dt.weekday().alias("weekday"),
        pl.col("receipt_time").dt.weekday().alias("day_of_week"),
    ])

    features = features.with_columns([
        (2 * np.pi * pl.col("hour")    / 24).sin().alias("hour_sin"),
        (2 * np.pi * pl.col("hour")    / 24).cos().alias("hour_cos"),
        (2 * np.pi * pl.col("weekday") / 7 ).sin().alias("day_sin"),
        (2 * np.pi * pl.col("weekday") / 7 ).cos().alias("day_cos"),
    ])

    features = features.with_columns([
        # Polars weekday: Mon=1 … Sun=7
        (pl.col("weekday") >= 6).cast(pl.Int8).alias("is_weekend"),
        pl.col("receipt_time").dt.date().cast(pl.Utf8)
          .is_in(holidays).cast(pl.Int8).alias("is_holiday"),
        (pl.col("receipt_time").dt.date().cast(pl.Utf8) == holiday_eve)
          .cast(pl.Int8).alias("is_holiday_eve"),
    ])

    features = features.with_columns(
        pl.col("order_id").cum_count().over("delivery_user_id")
          .alias("delivery_sequence")
    )
    return features


### 2.10 `add_workload_nonlinear` — Non-linear queue features

In [ ]:
def add_workload_nonlinear(
    features: pl.DataFrame,
    workload_cap: int = 20,
    high_load_thresh: int = 10,
    overload_thresh: int = 15,
    rank_cap: int = 15,
    late_batch_thresh: int = 8,
    extreme_batch_thresh: int = 12,
) -> pl.DataFrame:
    """
    Derive non-linear features from the ETA saturation curve.

    Saturation observed at ~15 concurrent orders across all cities
    (consistent with M/G/1 queuing: W ∝ ρ/(1-ρ)).
    Batch-rank effect is non-linear: monotonic then explodes at tail.

    Adds columns:
        workload_capped       — clipped at workload_cap
        high_load             — binary: active_orders > high_load_thresh
        overloaded            — binary: active_orders > overload_thresh
        batch_rank_capped     — clipped at rank_cap
        late_batch            — binary: batch_rank > late_batch_thresh
        extreme_batch         — binary: batch_rank > extreme_batch_thresh
    """
    features = features.with_columns([
        pl.col("active_orders_at_receipt_time")
          .clip(upper_bound=workload_cap).alias("workload_capped"),
        (pl.col("active_orders_at_receipt_time") > high_load_thresh)
          .cast(pl.Int8).alias("high_load"),
        (pl.col("active_orders_at_receipt_time") > overload_thresh)
          .cast(pl.Int8).alias("overloaded"),
        pl.col("batch_rank").clip(upper_bound=rank_cap).alias("batch_rank_capped"),
        (pl.col("batch_rank") > late_batch_thresh).cast(pl.Int8).alias("late_batch"),
        (pl.col("batch_rank") > extreme_batch_thresh).cast(pl.Int8).alias("extreme_batch"),
    ])
    return features


### 2.11 `add_gps_missingness_flag` — Structural fill + indicator

In [ ]:
def add_gps_missingness_flag(features: pl.DataFrame) -> pl.DataFrame:
    """
    Encode GPS absence as a first-class operational state.

    Strategy: structural fill (0) + binary indicator, NOT statistical imputation.
    Rationale: missing GPS ≠ stationary courier. It correlates with indoor
    deliveries, dense urban canyons, and device quality — structurally different
    from observed zero-speed movement.

    Diagnostic: ρ(workload, is_trajectory_available) ≈ 0.03 (HZ) / -0.07 (SH)
    — near-zero confirms the two signals are independent.

    Adds columns:
        is_trajectory_available  (0/1)
        fills speed_mean, distance_travelled, gps_points with 0 where null
    """
    features = features.with_columns(
        pl.col("speed_mean").is_not_null().cast(pl.Int8).alias("is_trajectory_available")
    )
    features = features.with_columns([
        pl.col("speed_mean").fill_null(0),
        pl.col("distance_travelled").fill_null(0),
        pl.col("gps_points").fill_null(0),
    ])

    corr = features.select(
        pl.corr("active_orders_at_receipt_time", "is_trajectory_available").alias("rho")
    ).item()
    print(f"  ρ(workload, GPS available) = {corr:.4f}  "
          f"[near-zero ✓ → independent signals]")
    return features


### 2.12 `add_typecode_encoding` — OHE grouping + frequency encoding

In [ ]:
def add_typecode_encoding(features: pl.DataFrame, top_n: int = 3) -> pl.DataFrame:
    """
    Dual encoding for the package type field.

    (a) OHE grouping — top_n codes → named groups; rest → 'rare'.
        One-hot encoded → binary columns for causal graph root nodes.
    (b) Frequency encoding — each code replaced by its count in the dataset.
        Dense ordinal signal for tree models and bandit policies.

    Adds columns:
        typecode_grouped          (categorical: 'type 1', 'type 2', 'type 3', 'rare')
        typecode_grouped_type 1   (binary OHE)
        typecode_grouped_type 2   (binary OHE)
        typecode_grouped_type 3   (binary OHE)
        typecode_cb               (frequency encoding)
    """
    # (a) Group
    top_codes = (
        features.select(pl.col("typecode").value_counts(sort=True))
        .unnest("typecode")
        .head(top_n)
        .get_column("typecode")
        .to_list()
    )
    mapping = {code: f"type {i+1}" for i, code in enumerate(top_codes)}

    features = features.with_columns(
        pl.col("typecode")
          .map_elements(lambda x: mapping.get(x, "rare"), return_dtype=pl.Utf8)
          .alias("typecode_grouped")
    )

    # OHE for top_n groups (exclude 'rare' as baseline)
    for i in range(1, top_n + 1):
        label = f"type {i}"
        features = features.with_columns(
            (pl.col("typecode_grouped") == label).cast(pl.Int8).alias(f"typecode_grouped_{label}")
        )

    # (b) Frequency encode
    freq = (
        features.group_by("typecode").len().rename({"len": "typecode_cb"})
    )
    features = features.join(freq, on="typecode", how="left")
    features = features.with_columns(pl.col("typecode_cb").fill_null(0))

    print(f"  typecode groups: {features['typecode_grouped'].value_counts().sort('typecode_grouped')}")
    return features


### 2.13 `add_spatial_congestion` — Grid-based SCI + courier local load

In [ ]:
def add_spatial_congestion(
    features: pl.DataFrame,
    grid_size: int = GRID_SIZE,
) -> pl.DataFrame:
    """
    Spatial Congestion Index (SCI) — adapted from Ke et al. (SIGKDD 2017).

    SCI(g, t) = number of deliveries in grid cell g during hour t.
    Tessellates the affine coordinate space into grid_size-unit cells.

    courier_local_load = delivery count per (courier, hour) — finer congestion.
    spatial_congestion_norm = z-score of SCI (makes it model-ready).

    Adds columns:
        grid_x, grid_y                — cell coordinates
        time_window                   — receipt_time truncated to hour
        spatial_congestion_index      — raw count
        spatial_congestion_norm       — z-score
        courier_local_load            — per-courier hourly count
    """
    features = features.with_columns([
        (pl.col("receipt_lng") // grid_size).alias("grid_x"),
        (pl.col("receipt_lat") // grid_size).alias("grid_y"),
        pl.col("receipt_time").dt.truncate("1h").alias("time_window"),
    ])

    sci = (
        features.group_by(["grid_x", "grid_y", "time_window"])
        .len().rename({"len": "spatial_congestion_index"})
    )
    features = features.join(sci, on=["grid_x", "grid_y", "time_window"], how="left")

    cll = (
        features.group_by(["delivery_user_id", "time_window"])
        .len().rename({"len": "courier_local_load"})
    )
    features = features.join(cll, on=["delivery_user_id", "time_window"], how="left")

    sci_mean = features["spatial_congestion_index"].mean()
    sci_std  = features["spatial_congestion_index"].std()
    features = features.with_columns(
        ((pl.col("spatial_congestion_index") - sci_mean) / sci_std)
        .alias("spatial_congestion_norm")
    )
    print(f"  SCI | mean: {sci_mean:.2f} | std: {sci_std:.2f} "
          f"| max: {features['spatial_congestion_index'].max()}")
    return features


### 2.14 `add_weather` — Temporal ASOF join with hourly weather

In [ ]:
def add_weather(
    features: pl.DataFrame,
    weather_csv: str,
    city_en: str,
) -> pl.DataFrame:
    """
    Join hourly weather observations to deliveries via ASOF backward join.
    Matches each receipt_time to the most recent prior weather record.

    Weather source: Open-Meteo hourly data (temperature, precipitation,
    wind speed, weather code, custom WSI composite).

    Adds columns (from weather CSV):
        temperature_2m, precipitation, windspeed_10m,
        WSI, norm_weathercode_sev
    """
    weather = (
        pl.read_csv(weather_csv)
        .with_columns(
            pl.col("datetime").str.strptime(pl.Datetime, strict=False).alias("weather_time")
        )
        .drop("datetime")
        .filter(pl.col("city") == city_en)
        .sort("weather_time")
    )

    features = features.sort("receipt_time")
    features = features.join_asof(
        weather,
        left_on="receipt_time",
        right_on="weather_time",
        strategy="backward",
    ).drop("city")

    pct_null = features["WSI"].null_count() / features.height * 100
    print(f"  Weather joined | WSI null: {pct_null:.1f}%")
    return features


### 2.15 `final_sanity_check` — Validation gates

In [ ]:
def final_sanity_check(features: pl.DataFrame, city_en: str) -> None:
    """
    Four-gate validation. Raises AssertionError on failure so the pipeline
    stops rather than silently saving a corrupt features file.
    """
    print(f"\n{'─'*55}")
    print(f"  Sanity check: {city_en}")
    print(f"{'─'*55}")

    # 1. No null targets
    null_eta = features["eta_mins"].null_count()
    assert null_eta == 0, f"FAIL: {null_eta} null eta_mins values!"
    print(f"  ✓ eta_mins null count: 0")

    # 2. Row uniqueness
    assert features.height == features["order_id"].n_unique(),         "FAIL: duplicate order_id rows detected!"
    print(f"  ✓ Row count == unique order_id count: {features.height:,}")

    # 3. No accidental join duplicates
    dup_cols = [c for c in features.columns if c.endswith("_right")]
    assert not dup_cols, f"FAIL: duplicate columns found: {dup_cols}"
    print(f"  ✓ No duplicate columns")

    # 4. Key feature null counts
    key_cols = ["active_orders_at_receipt_time", "batch_rank", "workload_capped",
                "hour_sin", "WSI"]
    for col in key_cols:
        if col in features.columns:
            n = features[col].null_count()
            status = "✓" if n == 0 else "⚠"
            print(f"  {status} {col} nulls: {n}")

    print(f"  Total columns: {len(features.columns)}")
    print(f"  Total rows:    {features.height:,}")
    print(f"{'─'*55}\n")


## 3. EDA / Diagnostic Functions

Quick visual diagnostics per city. Call these after `build_city_features()`
or from the cross-city comparison section.


In [ ]:
def plot_speed_ccdf(con, gps_path: str, city_en: str) -> None:
    """
    Log-log CCDF of courier speed.
    A heavy right tail confirms the trajectory captures real inter-zone movement.
    A sharp drop-off at the far end = rare cross-city reallocation events.
    """
    speed_df = con.execute(f"""
        WITH motion AS (
            SELECT
                SQRT(POWER(lat - LAG(lat) OVER w, 2) +
                     POWER(lng - LAG(lng) OVER w, 2)) AS dist,
                EXTRACT(EPOCH FROM (gps_time - LAG(gps_time) OVER w)) AS dt
            FROM parquet_scan('{gps_path}')
            WINDOW w AS (PARTITION BY postman_id ORDER BY gps_time)
        )
        SELECT dist / dt AS speed
        FROM motion
        WHERE dt > 0 AND dist IS NOT NULL AND dist / dt IS NOT NULL
    """).pl().sort("speed")

    n = speed_df.height
    ccdf = speed_df.with_row_index("rank").with_columns(
        (1 - pl.col("rank") / n).alias("ccdf")
    )
    s = ccdf["speed"].to_numpy()
    p = ccdf["ccdf"].to_numpy()
    mask = (s > 0) & (p > 0)

    plt.figure(figsize=(6, 4))
    plt.loglog(s[mask], p[mask])
    plt.xlabel("Speed (log scale)")
    plt.ylabel("P(Speed ≥ v) (log scale)")
    plt.title(f"Courier Speed CCDF — {city_en}")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


def plot_workload_saturation(features: pl.DataFrame, city_en: str) -> None:
    """
    Bar chart: mean ETA vs active orders (clipped at 20).
    Illustrates M/G/1 saturation curve — delay explodes near ρ→1.
    """
    plot_df = (
        features
        .with_columns(pl.col("active_orders_at_receipt_time").clip(upper_bound=20))
        .group_by("active_orders_at_receipt_time")
        .agg(pl.mean("eta_mins").alias("mean_eta"), pl.len().alias("n"))
        .sort("active_orders_at_receipt_time")
        .to_pandas()
    )
    plt.figure(figsize=(11, 4))
    sns.barplot(x="active_orders_at_receipt_time", y="mean_eta",
                data=plot_df, palette="viridis",
                hue="active_orders_at_receipt_time", legend=False)
    plt.axvline(x=14.5, color="red", linestyle="--", alpha=0.7, label="Saturation threshold")
    plt.title(f"Workload Saturation Curve — {city_en}")
    plt.xlabel("Active Orders at Receipt Time (clipped at 20)")
    plt.ylabel("Mean ETA (mins)")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_batch_rank_eta(features: pl.DataFrame, city_en: str) -> None:
    """
    Line chart: mean ETA vs batch rank (clipped at 15).
    Monotonic rise validates the late_batch binary feature.
    """
    plot_df = (
        features
        .with_columns(pl.col("batch_rank").clip(upper_bound=15))
        .group_by("batch_rank")
        .agg(pl.mean("eta_mins").alias("mean_eta"), pl.len().alias("n"))
        .sort("batch_rank")
        .to_pandas()
    )
    plt.figure(figsize=(9, 4))
    plt.plot(plot_df["batch_rank"], plot_df["mean_eta"], marker="o")
    plt.fill_between(plot_df["batch_rank"],
                     plot_df["mean_eta"] * 0.9, plot_df["mean_eta"] * 1.1,
                     alpha=0.15)
    plt.axvline(x=8, color="orange", linestyle="--", alpha=0.7, label="late_batch threshold")
    plt.title(f"Batch Rank vs Mean ETA — {city_en}")
    plt.xlabel("Batch Rank (clipped at 15)")
    plt.ylabel("Mean ETA (mins)")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_eta_distribution(features: pl.DataFrame, city_en: str, clip_pct: float = 0.95) -> None:
    """
    KDE of eta_mins clipped at the 95th percentile to suppress extreme outliers.
    """
    cap = features["eta_mins"].quantile(clip_pct)
    data = features.filter(pl.col("eta_mins") <= cap)["eta_mins"].to_numpy()
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data, fill=True)
    plt.axvline(data.mean(), color="red",  linestyle="--", label=f"Mean {data.mean():.0f}")
    plt.axvline(float(np.median(data)), color="green", linestyle="--",
                label=f"Median {float(np.median(data)):.0f}")
    plt.title(f"ETA Distribution (≤P{int(clip_pct*100)}) — {city_en}")
    plt.xlabel("ETA (mins)")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()


def print_correlation_table(features: pl.DataFrame, city_en: str) -> None:
    """
    Key Pearson correlations with eta_mins.
    """
    targets = [
        "workload_capped", "active_orders_at_receipt_time",
        "batch_rank", "batch_rank_capped",
        "speed_mean", "gps_gap_min",
        "pickup_destination_distance", "spatial_congestion_norm",
        "WSI", "hour_sin",
    ]
    rows = []
    for col in targets:
        if col in features.columns:
            rho = features.select(pl.corr(col, "eta_mins")).item()
            rows.append((col, f"{rho:+.4f}"))
    print(f"\nPearson ρ with eta_mins — {city_en}")
    print(f"{'Feature':<38} {'ρ':>8}")
    print("─" * 48)
    for feat, rho in sorted(rows, key=lambda x: abs(float(x[1])), reverse=True):
        print(f"  {feat:<36} {rho:>8}")


## 4. Master Pipeline Function

`build_city_features` orchestrates all steps for a single city config dict.


In [ ]:
def build_city_features(
    cfg: dict,
    base: str = BASE,
    weather_base: str = WEATHER_BASE,
    pre_min: int = PRE_MIN,
    max_gap: int = MAX_GAP,
    grid_size: int = GRID_SIZE,
    run_eda_plots: bool = True,
) -> pl.DataFrame:
    """
    End-to-end feature pipeline for one city.

    Parameters
    ----------
    cfg         : city config dict from CITY_CONFIGS
    base        : directory containing delivery CSVs and GPS parquets
    weather_base: directory containing weather CSVs
    pre_min     : GPS look-back window (minutes)
    max_gap     : GPS staleness threshold (minutes)
    grid_size   : SCI grid cell size (affine units)
    run_eda_plots: whether to show inline EDA plots

    Returns
    -------
    pl.DataFrame  final feature matrix with 'city' column added
    """
    city_en = cfg["city_en"]
    gps_path = base + cfg["gps_file"]

    print(f"\n{'='*60}")
    print(f"  Processing: {city_en}")
    print(f"{'='*60}")

    # 1. Load & normalise delivery CSV
    print("\n[1/10] Loading delivery data...")
    delivery = load_delivery(base + cfg["delivery_file"])

    # 2. Euclidean distance (operate on delivery before GPS joins)
    print("[2/10] Computing pickup-destination distance...")
    delivery = add_euclidean_distance(delivery)

    # 3. Batch size & rank
    print("[3/10] Computing batch features...")
    delivery = add_batch_features(delivery)

    # 4. Workload (event sweep)
    print("[4/10] Computing courier workload (event sweep)...")
    delivery = add_workload(delivery)

    # 5. GPS window + snapshot + staleness filter
    print("[5/10] Building GPS window & courier snapshot...")
    con = duckdb.connect()
    couriers   = delivery["delivery_user_id"].unique().to_list()
    start_time = delivery["receipt_time"].min()
    end_time   = delivery["receipt_time"].max()

    gps_df = load_gps_window(con, gps_path, couriers, start_time, end_time, pre_min)
    state  = courier_snapshot(delivery, gps_df)
    state  = filter_stale_gps(state, max_gap)

    # 6. Trajectory features
    print("[6/10] Extracting trajectory features (DuckDB)...")
    con.register("delivery_tbl", delivery.to_pandas())
    speed_cap    = compute_speed_percentile(con, gps_path)
    traj_features = compute_trajectory_features(con, gps_path, speed_cap, pre_min)

    # 7. Merge: delivery + state snapshot + trajectory
    print("[7/10] Merging feature tables...")
    features = (
        delivery
        .join(state.select(["order_id", "last_x", "last_y", "gps_gap_min"]),
              on="order_id", how="left")
        .join(traj_features, on="order_id", how="left")
    )

    # 8. Temporal & calendar
    print("[8/10] Adding temporal & calendar features...")
    features = add_temporal_features(features, cfg["holidays"], cfg["holiday_eve"])

    # 9. Non-linear workload/batch indicators
    features = add_workload_nonlinear(features)

    # 10. GPS missingness flag + structural fill
    print("[9/10] Adding GPS missingness indicator...")
    features = add_gps_missingness_flag(features)

    # 11. Typecode encoding
    print("[10/10] Encoding typecode...")
    features = add_typecode_encoding(features)

    # 12. Spatial congestion
    features = add_spatial_congestion(features, grid_size)

    # 13. Weather
    features = add_weather(
        features,
        weather_base + cfg["weather_file"],
        city_en,
    )

    # 14. Add city label
    features = features.with_columns(pl.lit(city_en).alias("city"))

    # 15. Validation
    final_sanity_check(features, city_en)

    # 16. Optional EDA
    if run_eda_plots:
        print(f"\n--- EDA Plots: {city_en} ---")
        plot_eta_distribution(features, city_en)
        plot_speed_ccdf(con, gps_path, city_en)
        plot_workload_saturation(features, city_en)
        plot_batch_rank_eta(features, city_en)
        print_correlation_table(features, city_en)

    con.close()
    return features


## 5. Run All Cities

Execute the full pipeline for every city in `CITY_CONFIGS`.  
Results are stored in `city_results` (dict) and optionally merged into
`all_cities_df` for cross-city analysis.


In [ ]:
# Set run_eda_plots=False for a silent batch run
city_results = {}

for cfg in CITY_CONFIGS:
    features = build_city_features(
        cfg,
        base=BASE,
        weather_base=WEATHER_BASE,
        pre_min=PRE_MIN,
        max_gap=MAX_GAP,
        grid_size=GRID_SIZE,
        run_eda_plots=True,       # ← flip to False for batch runs
    )
    city_results[cfg["city_en"]] = features

print("\n✅ All cities processed:")
for city, df in city_results.items():
    print(f"  {city}: {df.height:,} rows × {len(df.columns)} columns")


## 6. Save Outputs

Per-city CSVs (full feature matrix) and a slimmed-down 'selected features' CSV.
Also creates a combined parquet for cross-city analysis and causal discovery.


In [ ]:
import os

for city_en, features in city_results.items():
    slug = city_en.lower()

    # Full feature matrix
    full_path = os.path.join(OUTPUT_DIR, f"{slug}_final_weatheradded.csv")
    features.write_csv(full_path)
    print(f"  Saved full: {full_path}")

    # Selected features only
    available_sel = [c for c in SELECTED_FEATURES if c in features.columns]
    sel_path = os.path.join(OUTPUT_DIR, f"{slug}_selected_features.csv")
    features.select(available_sel).write_csv(sel_path)
    print(f"  Saved selected ({len(available_sel)} cols): {sel_path}")

print("\n✅ Per-city files saved.")


In [ ]:
# ── Combined parquet for cross-city analysis ──────────────────────────────────
# Aligns columns across cities (fills missing cols with null)

all_dfs = list(city_results.values())
combined = pl.concat(all_dfs, how="diagonal")   # diagonal = union of columns

combined_path = os.path.join(OUTPUT_DIR, "all_cities_combined.parquet")
combined.write_parquet(combined_path)
print(f"Combined dataset: {combined.height:,} rows × {combined.width} columns")
print(f"Saved to: {combined_path}")


## 7. Cross-City EDA

Overlay plots comparing all three cities — useful for thesis figures.


In [ ]:
def plot_eta_by_city(city_results: dict, clip_pct: float = 0.95) -> None:
    # ETA KDE overlay for all cities on one axis.
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
    for city, df in city_results.items():
        cap  = df["eta_mins"].quantile(clip_pct)
        data = df.filter(pl.col("eta_mins") <= cap)["eta_mins"].to_numpy()
        med  = float(np.median(data))
        sns.kdeplot(data, ax=ax, label=f"{city} (median={med:.0f} min)",
                    color=colors.get(city), fill=True, alpha=0.25)
    ax.set_xlabel("ETA (mins)")
    ax.set_ylabel("Density")
    ax.set_title(f"ETA Distribution by City (<=P{int(clip_pct*100)})")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_saturation_by_city(city_results: dict) -> None:
    # Workload saturation curves — 1x3 subplot grid.
    fig, axes = plt.subplots(1, len(city_results), figsize=(15, 4), sharey=False)
    colors = {"Shanghai": "#e74c3c", "Hangzhou": "#2980b9", "Chongqing": "#27ae60"}
    for ax, (city, df) in zip(axes, city_results.items()):
        plot_df = (
            df.with_columns(pl.col("active_orders_at_receipt_time").clip(upper_bound=20))
            .group_by("active_orders_at_receipt_time")
            .agg(pl.mean("eta_mins").alias("mean_eta"))
            .sort("active_orders_at_receipt_time")
            .to_pandas()
        )
        ax.bar(plot_df["active_orders_at_receipt_time"], plot_df["mean_eta"],
               color=colors.get(city, "steelblue"), alpha=0.8)
        ax.axvline(x=14.5, color="red", linestyle="--", alpha=0.6)
        ax.set_title(city)
        ax.set_xlabel("Active Orders (clipped at 20)")
        ax.set_ylabel("Mean ETA (mins)")
    plt.suptitle("Workload Saturation Curve — All Cities", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


def plot_feature_corr_heatmap(city_results: dict) -> None:
    # Side-by-side correlation heatmaps (key features vs eta_mins).
    key_features = [
        "workload_capped", "batch_rank_capped", "speed_mean",
        "pickup_destination_distance", "spatial_congestion_norm",
        "gps_gap_min", "WSI", "hour_sin", "is_weekend", "is_holiday",
        "eta_mins",
    ]
    fig, axes = plt.subplots(1, len(city_results), figsize=(18, 8))
    for ax, (city, df) in zip(axes, city_results.items()):
        avail = [c for c in key_features if c in df.columns]
        corr_mat = df.select(avail).corr().to_pandas()
        corr_mat.columns = avail
        corr_mat.index   = avail
        sns.heatmap(corr_mat, ax=ax, cmap="coolwarm", center=0,
                    annot=True, fmt=".2f", annot_kws={"size": 7},
                    linewidths=0.4, vmin=-1, vmax=1)
        ax.set_title(city)
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)
    plt.suptitle("Pearson Correlation Heatmap — Key Features", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


# ── Run cross-city plots ──────────────────────────────────────────────────────
plot_eta_by_city(city_results)
plot_saturation_by_city(city_results)
plot_feature_corr_heatmap(city_results)


## 8. Feature Summary Table

Print a compact summary of all features for each city — useful for the thesis.


In [ ]:
def feature_summary(city_results: dict) -> None:
    for city, df in city_results.items():
        print(f"\n{'─'*60}")
        print(f"  {city}  |  {df.height:,} rows × {len(df.columns)} cols")
        print(f"{'─'*60}")
        num_cols = [c for c, t in df.schema.items()
                    if isinstance(t, (pl.Float32, pl.Float64, pl.Int8,
                                     pl.Int16, pl.Int32, pl.Int64,
                                     pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64))]
        stats = df.select(num_cols).describe().to_pandas().set_index("statistic")
        eta_row = stats.loc["mean"]
        null_counts = df.select([pl.col(c).null_count().alias(c) for c in num_cols])
        print(f"  eta_mins  mean={df['eta_mins'].mean():.1f}  "
              f"median={df['eta_mins'].median():.1f}  "
              f"p95={df['eta_mins'].quantile(0.95):.1f}")
        for col in sorted(num_cols):
            n_null = null_counts[col].item()
            tag = f"  ⚠ {n_null} nulls" if n_null > 0 else ""
            print(f"  {col:<40}{tag}")

feature_summary(city_results)
